In [1]:


import pandas as pd
from datetime import datetime
from pathlib import Path


In [6]:

# Path to Excel file (same folder)
file_path = Path("stationlist_clean.csv")  # change filename if needed

# Read Excel
stations_df = pd.read_csv(file_path)

# Filter NEW BRUNSWICK stations
nb_stations = stations_df.loc[
    stations_df["Province"] == "NEW BRUNSWICK",
    "Climate ID"
].astype(str).unique()


In [7]:

province_code = "NB"
start_year = 1995
end_year = datetime.now().year

base_url = "https://dd.weather.gc.ca/today/climate/observations/daily/csv/"


In [8]:
dfs = []

for climate_id in nb_stations:
    for year in range(start_year, end_year + 1):
        url = (
            f"{base_url}/{province_code}/"
            f"climate_daily_{province_code}_{climate_id}_{year}_P1D.csv"
        )
        try:
            df = pd.read_csv(
                url,
                encoding="latin-1",
                dtype={"Climate ID": "string"},
                low_memory=False
            )
            dfs.append(df)
        except Exception:
            # File does not exist (station not active that year)
            continue

In [9]:
NB_daily_1995_present = pd.concat(dfs, ignore_index=True)

In [10]:

NB_daily_1995_present["Date/Time"] = pd.to_datetime(
    NB_daily_1995_present["Date/Time"]
)

NB_daily_1995_present["Station Name"] = (
    NB_daily_1995_present["Station Name"].astype("string")
)


In [11]:

NB_daily_1995_present["Spd of Max Gust (km/h)"] = pd.to_numeric(
    NB_daily_1995_present["Spd of Max Gust (km/h)"],
    errors="coerce"
)


In [12]:
NB_daily_1995_present.info()

<class 'pandas.DataFrame'>
RangeIndex: 203443 entries, 0 to 203442
Data columns (total 31 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Longitude (x)              203443 non-null  float64       
 1   Latitude (y)               203443 non-null  float64       
 2   Station Name               203443 non-null  string        
 3   Climate ID                 203443 non-null  string        
 4   Date/Time                  203443 non-null  datetime64[us]
 5   Year                       203443 non-null  int64         
 6   Month                      203443 non-null  int64         
 7   Day                        203443 non-null  int64         
 8   Data Quality               26505 non-null   object        
 9   Max Temp (°C)              168118 non-null  float64       
 10  Max Temp Flag              3326 non-null    object        
 11  Min Temp (°C)              168585 non-null  float64       
 12 

In [13]:
NB_daily_1995_present["Year"].min(), NB_daily_1995_present["Year"].max()

(1995, 2026)

In [16]:
NB_daily_1995_present.head(10)

,Longitude (x),Latitude (y),Station Name,Climate ID,Date/Time,Year,Max Temp (°C),Min Temp (°C),Mean Temp (°C),Heat Deg Days (°C),Cool Deg Days (°C),Total Rain (mm),Total Snow (cm),Total Precip (mm),Snow on Grnd (cm),Dir of Max Gust (10s deg),Spd of Max Gust (km/h)
0,-67.72,46.71,AROOSTOOK,8100300,1995-01-01,1995,-8.0,-14.0,-11.0,29.0,0.0,0.0,1.2,1.2,32.0,NaN,NaN
1,-67.72,46.71,AROOSTOOK,8100300,1995-01-02,1995,-4.0,-9.0,-6.5,24.5,0.0,0.0,1.8,1.8,33.0,NaN,NaN
2,-67.72,46.71,AROOSTOOK,8100300,1995-01-03,1995,-6.0,-10.5,-8.3,26.3,0.0,0.0,0.0,0.0,34.0,NaN,NaN
3,-67.72,46.71,AROOSTOOK,8100300,1995-01-04,1995,-3.0,-19.5,-11.3,29.3,0.0,0.0,2.2,2.2,34.0,NaN,NaN
4,-67.72,46.71,AROOSTOOK,8100300,1995-01-05,1995,-9.0,-21.0,-15.0,33.0,0.0,0.0,0.0,0.0,35.0,NaN,NaN
5,-67.72,46.71,AROOSTOOK,8100300,1995-01-06,1995,-3.0,-14.0,-8.5,26.5,0.0,0.0,10.2,10.2,35.0,NaN,NaN
6,-67.72,46.71,AROOSTOOK,8100300,1995-01-07,1995,-6.0,-18.0,-12.0,30.0,0.0,0.0,6.6,6.6,45.0,NaN,NaN
7,-67.72,46.71,AROOSTOOK,8100300,1995-01-08,1995,-6.5,-20.0,-13.3,31.3,0.0,0.0,0.0,0.0,46.0,NaN,NaN
8,-67.72,46.71,AROOSTOOK,8100300,1995-01-09,1995,-4.5,-21.0,-12.8,30.8,0.0,0.0,1.0,1.0,46.0,NaN,NaN
9,-67.72,46.71,AROOSTOOK,8100300,1995-01-10,1995,-20.0,-30.0,-25.0,43.0,0.0,0.0,0.0,0.0,45.0,NaN,NaN


In [15]:
NB_daily_1995_present =NB_daily_1995_present.drop(
    columns=[
        'Month', 'Day',
        'Data Quality',
        'Max Temp Flag', 'Min Temp Flag', 'Mean Temp Flag',
        'Heat Deg Days Flag', 'Cool Deg Days Flag', 'Total Rain Flag',
        'Total Snow Flag', 'Total Precip Flag', 'Snow on Grnd Flag',
       'Spd of Max Gust Flag',  'Dir of Max Gust Flag']
)